In [6]:
import pandas as pd
import os
import re
import numpy as np
from tqdm import tqdm

In [2]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

2.5.1+cu121
GPU available: True
GPU name: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
from sentence_transformers import SentenceTransformer

c:\Users\Akash Ghosal\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


part 1

In [17]:
data = pd.read_pickle("../data/processed/final_dataset.pkl")
print(data.shape)
data.head()

(14648, 5)


,ticker,date,transcript,ret_1m,ret_3m
0,BILI,2020-08-27,"Prepared Remarks:\nOperator\nGood day, and wel...",-0.073770,0.373726
1,GFF,2020-07-30,Prepared Remarks:\nOperator\nThank you for sta...,0.105072,0.064720
2,LRCX,2019-10-23,Prepared Remarks:\nOperator\nGood day and welc...,0.131744,0.320272
3,BBSI,2019-11-06,"Prepared Remarks:\nOperator\nGood day, everyon...",-0.081740,-0.096723
4,CSTE,2019-08-07,Prepared Remarks:\nOperator\nGreetings and wel...,-0.068653,0.102332


In [18]:
data['transcript_length'] = data['transcript'].str.len()
data['transcript_length'].describe()

count     14648.000000
mean      48116.601925
std       14631.407794
min        3682.000000
25%       37994.250000
50%       48717.000000
75%       57560.750000
max      188514.000000
Name: transcript_length, dtype: float64

In [ ]:
# cleaning
def clean_text(text):
    text = text.lower()

    # remove everything from "prepared remarks" to first operator greeting
    text = re.sub(r'prepared remarks:.*?operator', ' ', text, flags=re.DOTALL)

    # remove line breaks
    text = re.sub(r'\n', ' ', text)

    # remove non-alphanumeric characters
    text = re.sub(r'[^a-zA-Z0-9 .]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

data['clean_transcript'] = data['transcript'].apply(clean_text)
data[['transcript','clean_transcript']].head(10)

,transcript,clean_transcript
0,"Prepared Remarks:\nOperator\nGood day, and wel...",good day and welcome to the bilibili 2020 seco...
1,Prepared Remarks:\nOperator\nThank you for sta...,thank you for standing by. this is the confere...
2,Prepared Remarks:\nOperator\nGood day and welc...,good day and welcome to the september 2019 qua...
3,"Prepared Remarks:\nOperator\nGood day, everyon...",good day everyone and thank you for participat...
4,Prepared Remarks:\nOperator\nGreetings and wel...,greetings and welcome to the caesarstone limit...
5,"Prepared Remarks:\nOperator\nGood afternoon, a...",good afternoon and welcome to the green dot th...
6,Prepared Remarks:\nOperator\nLadies and gentle...,ladies and gentlemen thank you for standing by...
7,Prepared Remarks:\nOperator\nLadies and gentle...,ladies and gentlemen thank you for standing by...
8,Prepared Remarks:\nOperator\nGreetings and wel...,greetings and welcome to the abm industries se...
9,Prepared Remarks:\nOperator\nGood morning and ...,good morning and welcome to super group s seco...


In [27]:
# chuncking transcripts
def chunk_text(text, chunk_size=400):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

In [33]:
# test
sample_text = data['clean_transcript'].iloc[0]
chunks = chunk_text(sample_text)

print("Number of chunks:", len(chunks))
print("First chunk preview:\n")
print(chunks[0][:500])

Number of chunks: 15
First chunk preview:

good day and welcome to the bilibili 2020 second quarter earnings conference call. today s conference is being recorded. at this time i would like to turn the conference over to juliet yang senior director of investor relations. please go ahead. juliet yang senior director of investor relations thank you operator. please note the discussion today will contain forward looking statements relating to the company s future performance and are intended to qualify for the safe harbor from liability as 


In [34]:
# chunck for all transcripts and create a dataset having all the chuncks in their perspective row
chunk_rows = []

for i, row in data.iterrows():
    chunks = chunk_text(row['clean_transcript'])
    
    for chunk in chunks:
        chunk_rows.append({
            "ticker": row['ticker'],
            "date": row['date'],
            "ret_1m": row['ret_1m'],
            "ret_3m": row['ret_3m'],
            "chunk_text": chunk
        })

chunk_df = pd.DataFrame(chunk_rows)

print("Chunk dataset shape:", chunk_df.shape)
chunk_df.head(10)

Chunk dataset shape: (308383, 5)


,ticker,date,ret_1m,ret_3m,chunk_text
0,BILI,2020-08-27,-0.07377,0.373726,good day and welcome to the bilibili 2020 seco...
1,BILI,2020-08-27,-0.07377,0.373726,maus were 172 million up 55 and daus were up 5...
2,BILI,2020-08-27,-0.07377,0.373726,working well for us and we are determined to c...
3,BILI,2020-08-27,-0.07377,0.373726,daily life of the immortal king xian wang de r...
4,BILI,2020-08-27,-0.07377,0.373726,stronger. in the second quarter we were excite...
5,BILI,2020-08-27,-0.07377,0.373726,industry we have seen more hosts and talent ag...
6,BILI,2020-08-27,-0.07377,0.373726,s remarks. i will now provide a brief overview...
7,BILI,2020-08-27,-0.07377,0.373726,deposits as well as short term investments of ...
8,BILI,2020-08-27,-0.07377,0.373726,the ratio has significantly increased compared...
9,BILI,2020-08-27,-0.07377,0.373726,marketing campaign trilogy profits with hou la...


In [35]:
chunk_df.to_pickle("../data/processed/chunk_dataset.pkl")
print("Chunk dataset saved")

Chunk dataset saved


part 2

In [4]:
chunk_df = pd.read_pickle("../data/processed/chunk_dataset.pkl")

In [5]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

print("Model loaded on GPU")

c:\Users\Akash Ghosal\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Akash Ghosal\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Akash Ghosal\AppData\Local\Programs\Python\Pytho

Model loaded on GPU


In [7]:
# prepare texts for embedding
texts = chunk_df["chunk_text"].tolist()
print("Total chunks:", len(texts))

Total chunks: 308383


In [8]:
# batch embedding
batch_size = 256

embeddings = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_emb = model.encode(
        batch,
        show_progress_bar=False,
        convert_to_numpy=True
    )
    embeddings.append(batch_emb)

embeddings = np.vstack(embeddings)

print("Embedding shape:", embeddings.shape)


100%|██████████| 1205/1205 [21:50<00:00,  1.09s/it]


Embedding shape: (308383, 384)


In [9]:
np.save("../data/processed/chunk_embeddings.npy", embeddings)
print("Embeddings saved")

Embeddings saved


In [10]:
chunk_df["embedding"] = list(embeddings)
chunk_df.head(10)

,ticker,date,ret_1m,ret_3m,chunk_text,embedding
0,BILI,2020-08-27,-0.07377,0.373726,good day and welcome to the bilibili 2020 seco...,"[-0.047811184, 0.032338817, 0.012920213, 0.003..."
1,BILI,2020-08-27,-0.07377,0.373726,maus were 172 million up 55 and daus were up 5...,"[-0.010296496, -0.023134926, -0.027121585, -0...."
2,BILI,2020-08-27,-0.07377,0.373726,working well for us and we are determined to c...,"[0.030230569, -0.120629355, -0.022526504, -0.0..."
3,BILI,2020-08-27,-0.07377,0.373726,daily life of the immortal king xian wang de r...,"[-0.0642399, -0.0075253937, -0.016872097, -0.0..."
4,BILI,2020-08-27,-0.07377,0.373726,stronger. in the second quarter we were excite...,"[-0.04670263, -0.078211054, 0.02346434, -0.078..."
5,BILI,2020-08-27,-0.07377,0.373726,industry we have seen more hosts and talent ag...,"[0.013950001, -0.07429011, -0.055840347, -0.12..."
6,BILI,2020-08-27,-0.07377,0.373726,s remarks. i will now provide a brief overview...,"[-0.018213462, -0.030614529, -0.004643278, -0...."
7,BILI,2020-08-27,-0.07377,0.373726,deposits as well as short term investments of ...,"[-0.009643882, -0.027875183, -0.022842336, -0...."
8,BILI,2020-08-27,-0.07377,0.373726,the ratio has significantly increased compared...,"[0.03846791, -0.12595412, -0.010404574, -0.057..."
9,BILI,2020-08-27,-0.07377,0.373726,marketing campaign trilogy profits with hou la...,"[-0.014105241, -0.017782817, 0.047779754, -0.1..."


In [19]:
def average_embeddings(emb_list):
    return np.mean(emb_list, axis=0)

transcript_df = (
    chunk_df
    .groupby(["ticker","date","ret_1m","ret_3m"])["embedding"]
    .apply(list)
    .reset_index()
)

transcript_df["transcript_embedding"] = transcript_df["embedding"].apply(average_embeddings)

transcript_df.head(10)

,ticker,date,ret_1m,ret_3m,embedding,transcript_embedding
0,A,2020-02-18,-0.187758,-0.010816,"[[-0.06949754, 0.0065690754, -0.049980737, -0....","[-0.04184066, -0.024315318, 0.03424357, -0.013..."
1,A,2020-05-21,0.096347,0.213440,"[[-0.07158803, 0.03356093, -0.0500009, -0.0016...","[-0.042391513, -0.02445699, 0.031295378, -0.01..."
2,A,2020-08-18,0.017780,0.128846,"[[-0.07360973, 0.010902943, -0.08169886, -0.02...","[-0.027920121, -0.040649388, 0.021861328, -0.0..."
3,A,2020-11-23,0.045361,0.089923,"[[-0.0571277, 0.03701485, -0.066427134, -0.019...","[-0.030878326, -0.029609961, 0.023083668, -0.0..."
4,A,2021-02-16,-0.042439,0.018429,"[[-0.07734573, 0.022624558, -0.06453705, 0.000...","[-0.043922257, -0.039932363, 0.023395421, -0.0..."
5,A,2021-05-25,0.100878,0.281950,"[[-0.06555217, 0.010962941, -0.06778346, 0.011...","[-0.032605026, -0.041846197, 0.007577628, -0.0..."
6,A,2021-08-17,0.076875,-0.018231,"[[-0.091275536, -0.012989199, -0.0782016, -0.0...","[-0.029899646, -0.046073418, 0.00916201, -0.01..."
7,A,2022-02-22,0.028278,-0.033046,"[[-0.09183872, 0.015853096, -0.059686065, -0.0...","[-0.042260293, -0.041143212, 0.02063947, -0.01..."
8,A,2022-08-16,0.002937,0.104416,"[[-0.084358305, 0.014217644, -0.05901706, -0.0...","[-0.04646341, -0.04290449, 0.009335367, -0.019..."
9,AA,2021-01-20,0.012697,0.500000,"[[-0.069124594, 0.056654856, 0.009784603, 0.02...","[-0.033960667, 0.00011407037, 0.0027194251, 0...."


In [20]:
transcript_df = transcript_df.drop(columns=["embedding"])
transcript_df.head()

,ticker,date,ret_1m,ret_3m,transcript_embedding
0,A,2020-02-18,-0.187758,-0.010816,"[-0.04184066, -0.024315318, 0.03424357, -0.013..."
1,A,2020-05-21,0.096347,0.213440,"[-0.042391513, -0.02445699, 0.031295378, -0.01..."
2,A,2020-08-18,0.017780,0.128846,"[-0.027920121, -0.040649388, 0.021861328, -0.0..."
3,A,2020-11-23,0.045361,0.089923,"[-0.030878326, -0.029609961, 0.023083668, -0.0..."
4,A,2021-02-16,-0.042439,0.018429,"[-0.043922257, -0.039932363, 0.023395421, -0.0..."


In [21]:
print(transcript_df.shape)

(13715, 5)


In [22]:
transcript_df.to_pickle("../data/processed/transcript_embeddings.pkl")
print("Transcript embeddings saved")

Transcript embeddings saved
